# 📋 QLT02 · M02 — 검사 측정치 결측치 정제, 납품업체별 불량 이력 인코딩 전처리 모듈 구현

**학생용 독립 미션 Notebook · 교관 정답 미포함 | 필수 | CMS 2시간**

---

## 📌 0. 시작 전에

1. 이 파일을 Google Colab에서 엽니다. 첫 번째 코드 셀부터 순서대로 실행하세요.
2. 실습 자료는 준비 셀에 내장되어 있으므로 별도 ZIP을 업로드하지 않습니다.
3. 이 파일은 한 미션의 작성 도구이며, 이전 미션의 실제 산출물이 자동으로 이어졌다고 가정하지 마세요.
4. 실습 중 작성한 코드·대화·보고서는 백업해야 합니다. 실습용 PASS와 교관 승인은 다릅니다.

## 📌 오늘 할 일


### 지금 해결할 문제

수령 검사 데이터에 누락된 값, 중복 기록, 잘못된 형식이 섞여 있으면 이후 품질 예측 모델의 결과도 달라집니다. 여러분은 교육용 가상 품질 데이터 분석 담당자입니다. 잘못된 기록을 정리하고, 수정 이유를 남기며, 미래 정보가 분석에 섞이지 않도록 검증합니다. 실제 군수품 검사나 업체 평가, 합격 승인을 수행하는 과제는 아닙니다.

### 시작 순서

자료 준비: 프로젝트 공통 학생용 Notebook과 실습 ZIP을 준비하고, 미션 2의 학습 안내를 읽습니다. 정제 규칙 확인: 중복 로트, 업체 코드, 날짜·재고번호 형식, 허용 범위 오류 등 10종의 정제 규칙을 확인합니다. 코드 구현: 실습 프로젝트의 데이터 전처리 코드에서 핵심 정제 함수를 직접 구현하고 첫 실행 결과를 보존합니다.

추가 검증: KNN 이웃 수 3·5·10을 동일한 학습·검증 조건에서 비교합니다. 긴급 불출 예외 기록이 보존되는지, 업체 이력 계산에 미래 정보가 섞이지 않는지도 확인합니다. AI 검토 및 보완: AI가 제안한 설명이나 코드 선택 3건 이상을 제공 자료와 실제 처리 로그에 대조합니다. 수정 이유와 재실행 결과를 기록합니다.

결과 저장: 미션 2 작성 양식과 정제 품질 보고서에 실제 AI 요청·답변, 정제 전후 결과, 최종 판단을 기록합니다.

### 완료 전 확인

정제 규칙 10종 적용 여부, 정합성 검증 통과율 100% 달성 여부, 특수 예외 기록 보존 및 업체 이력의 정보 누수 방지 여부를 실제 실행 결과로 확인합니다. 작성 완료, 코드 실행 성공, 평가 기준 달성, 실제 교관 승인은 서로 구분합니다. 원본 예상 시간과 CMS 편성 시간은 미션 학습자료에서 확인하며, 실제 소요시간은 별도로 기록합니다.




## 🎯 미션 2 — 상황과 수행 안내

**학습할 문제:** 지금 해결할 문제 수령 검사 데이터에 누락된 값, 중복 기록, 잘못된 형식이 섞여 있으면 이후 품질 예측 모델의 결과도 달라집니다. 여러분은 교육용 가상 품질 데이터 분석 담당자입니다. 잘못된 기록을 정리하고, 수정 이유를 남기며, 미래 정보가 분석에 섞이지 않도록 검증합니다. 실제 군수품 검사나 업체 평가, 합격 승인을 수행하는 과제는 아닙니다.

### 진행 순서

1. **수행 범위와 검증 계획 정하기**
2. **핵심 구현·첫 실행과 추가 변경**
3. **근거 대조·보완·재실행**
4. **최종 판단·산출물 제출**

### 작성·검증·저장

- 이 파일은 **미션별 독립 Notebook**입니다. 첫 번째 준비 코드 셀로 가상 실습자료를 자동 준비합니다. 별도 ZIP 업로드는 필요하지 않습니다.
- 원본 미션 자료의 수치·평가 기준에 맞춰 함수 코드를 직접 작성하고, 첫 실행과 수정 후 실행을 구분합니다.
- 실제 AI 질문·답변, 자료 및 실행 결과 대조 3건 이상, 수정 이유를 해당 미션 Forms에 기록합니다.
- Notebook의 검토용 ZIP은 **작업 보관**용으로, 학습 PASS 또는 교관 승인이 아닙니다.


In [ ]:
#@title 실습자료 자동 준비 (클릭하여 실행) { display-mode: "form" }
# QLT02 v2: 내장 실습 ZIP 자동 준비 (인터넷 다운로드 및 수동 업로드 불필요)
from pathlib import Path
import base64, io, hashlib, zipfile, sys, importlib
PACKAGE_SHA256 = "973c12cd119b869beadb77ce8f0225516a7da9d03ae64ad368124c7742afdc28"
PACKAGE_B64 = "..."
# 학습자가 이미 작업한 파일은 절대 덮어쓰지 않습니다.
try:
    from google.colab import files as colab_files
    DESTINATION = Path('/content')
except ImportError:
    colab_files = None
    DESTINATION = Path.cwd()
ROOT = DESTINATION / 'QLT02_project'

def prepare_project():
    if ROOT.exists():
        if not (ROOT / 'pipeline/run_all.py').is_file():
            raise RuntimeError('기존 프로젝트 폴더가 불완전합니다. 백업 후 폴더를 확인하세요.')
        print('기존 작업 보존: 프로젝트 ZIP을 다시 풀지 않습니다.')
        return
    binary = base64.b64decode(PACKAGE_B64, validate=True)
    if hashlib.sha256(binary).hexdigest() != PACKAGE_SHA256:
        raise ValueError('내장 실습 ZIP의 무결성 검증 실패')
    with zipfile.ZipFile(io.BytesIO(binary)) as z:
        bad = z.testzip()
        if bad: raise ValueError('ZIP 오류: ' + bad)
        members = z.infolist()
        for item in members:
            p = Path(item.filename)
            if p.is_absolute() or not p.parts or p.parts[0] != 'QLT02_project' or '..' in p.parts:
                raise ValueError('위험한 ZIP 내부 경로: ' + item.filename)
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크는 허용하지 않습니다')
        z.extractall(DESTINATION)
    print('내장 실습 ZIP 검증 및 준비 완료:', ROOT)
prepare_project()
for p in [ROOT, ROOT / 'src']:
    if str(p) not in sys.path: sys.path.insert(0, str(p))
for name in ['numpy','pandas','scipy','sklearn','matplotlib']:
    try:
        mod = importlib.import_module(name)
        print(name, mod.__version__)
    except ImportError:
        print('라이브러리 부족:', name, '— 아래 설치 안내를 확인하세요')
print('프로젝트 위치:', ROOT)


라이브러리가 없는 경우에만 다음 셀의 주석을 해제해 설치하고 런타임을 재시작합니다. 설치 후 첫 준비 셀부터 다시 실행하세요.


In [ ]:
# 필요할 때만 실행
# %pip install -r /content/QLT02_project/requirements.txt

## 함수 구현 저장 방법

아래 helper를 실행합니다. 각 미션의 함수 정의를 AI와 작성하고 implementation 문자열에 전체 def를 넣습니다. 함수 이름·인자는 제공된 정의를 유지하세요. 저장 시 이전 파일을 .before로 보존합니다.


In [ ]:
from tools.patch_function import replace_function
import json
TASKS=json.loads((ROOT/"implementation_tasks.json").read_text())
SELECTED_MISSIONS=list(range(1,11)) # 선택 M4·M6·M8 미수행 시 여기에서 제외하고 forms에 사유를 남깁니다.


## 작업 기록·진행 현황·JSON 백업/복원

첫 준비 셀과 기본 helper 셀을 실행한 뒤 아래 도구를 사용하세요.

- `show_progress()` : 미션별 실행 기록의 존재를 보여줍니다. **학습 PASS나 교관 승인 여부는 판단하지 않습니다.**
- `save_mission_note(1, "...", "...", "...")` : 미션별 본인 의견·AI 대화·검증 근거를 별도의 메모에 보관합니다. 원래 `forms/` 작성 양식도 반드시 완성하세요.
- `export_work_json()` : 작업 파일을 JSON으로 내려받습니다. 특히 Colab 종료 전에 실행하세요.
- `restore_work_json()` : 백업 JSON을 선택해 작업을 복원합니다. 복원 전 별도 백업을 권장하며, 기본적으로 기존 파일은 덮어쓰지 않습니다.

JSON은 학습자의 코드·문서·실행 기록을 복사하는 **백업**이며, 실제 평가 결과나 교관 승인을 생성하지 않습니다. AI 대화는 본인이 기록한 내용만 보존됩니다.


In [ ]:
from pathlib import Path
import json, base64, hashlib, datetime, re
WORK_DIRS = ('src','pipeline','forms','docs','reports','notebooks','outputs','workbook_notes','data')
WORK_FILES = ('implementation_tasks.json',)
NOTE_ROOT = ROOT/'workbook_notes'
NOTE_ROOT.mkdir(exist_ok=True)

def _safe_work_path(relative):
    p=Path(relative)
    if p.is_absolute() or '..' in p.parts or not p.parts:
        raise ValueError('올바르지 않은 상대 경로')
    if p.parts[0] not in WORK_DIRS and str(p) not in WORK_FILES:
        raise ValueError('백업 허용 경로가 아닙니다: '+str(p))
    dest=(ROOT/p).resolve()
    if not dest.is_relative_to(ROOT.resolve()):
        raise ValueError('프로젝트 외부 경로')
    return dest

def show_progress():
    print('미션 | 실행기록 | 본인메모 | 참고: 결과 PASS/교관승인은 별도 확인')
    for n in range(1,11):
        logs=list((ROOT/'outputs'/f'M{n:02d}').glob('*/run.json'))
        note=NOTE_ROOT/f'M{n:02d}.md'
        print(f'M{n:02d}   | {len(logs)}건      | {"있음" if note.exists() else "없음"}')
    print('실행기록이 있어도 정량 기준·평가 PASS·실제 교관 승인을 의미하지 않습니다.')

def save_mission_note(mission, decision, ai_conversation, evidence):
    if not isinstance(mission,int) or not 1<=mission<=10:
        raise ValueError('미션 번호는 1～10')
    if not all(isinstance(v,str) for v in (decision,ai_conversation,evidence)):
        raise TypeError('내용은 문자열로 입력하세요')
    p=NOTE_ROOT/f'M{mission:02d}.md'
    with p.open('a',encoding='utf-8') as f:
        f.write('\n\n## '+datetime.datetime.now(datetime.timezone.utc).isoformat()+' 기록\n')
        f.write('\n### 본인 판단 및 수정 이유\n'+decision+'\n')
        f.write('\n### 실제 AI 요청·응답(원문만 기록)\n'+ai_conversation+'\n')
        f.write('\n### 제공 자료·실행 결과와의 대조 근거\n'+evidence+'\n')
    print('메모 저장:',p,'— forms 작성 양식도 별도로 완성하세요')
    return p

def _work_payload():
    records=[]
    for d in WORK_DIRS:
        folder=ROOT/d
        if not folder.exists():continue
        for path in sorted(folder.rglob('*')):
            if not path.is_file() or '__pycache__' in path.parts or path.is_symlink():continue
            relative=path.relative_to(ROOT).as_posix()
            data=path.read_bytes()
            if len(data)>25_000_000:
                print('대용량 백업 제외:',relative);continue
            records.append({'path':relative,'sha256':hashlib.sha256(data).hexdigest(),
                            'content_base64':base64.b64encode(data).decode('ascii')})
    return {'format':'QLT02_WORK_V2','created_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'note':'작업 백업일 뿐 PASS 또는 교관 승인 아님', 'files':records}

def export_work_json():
    out=ROOT.parent/'QLT02_work_backup.json'
    data=_work_payload()
    out.write_text(json.dumps(data,ensure_ascii=False),encoding='utf-8')
    print('JSON 백업:',len(data['files']),'파일',out)
    if colab_files is not None:colab_files.download(str(out))
    return out

def restore_work_json(backup_path=None, overwrite=False):
    if backup_path is None:
        if colab_files is None:raise ValueError('로컬 실행 시 백업 JSON 파일 경로를 지정하세요')
        selected=colab_files.upload()
        if len(selected)!=1:raise ValueError('작업 백업 JSON 파일 하나만 선택하세요')
        backup_path=next(iter(selected))
    obj=json.loads(Path(backup_path).read_text(encoding='utf-8'))
    if obj.get('format')!='QLT02_WORK_V2' or not isinstance(obj.get('files'),list):
        raise ValueError('QLT02 v2 백업 JSON 형식이 아닙니다')
    if len(obj['files'])>2000:raise ValueError('파일 수 제한 초과')
    staged=[]
    for item in obj['files']:
        dest=_safe_work_path(item['path'])
        raw=base64.b64decode(item['content_base64'],validate=True)
        if len(raw)>25_000_000 or hashlib.sha256(raw).hexdigest()!=item['sha256']:
            raise ValueError('백업 무결성 오류: '+item['path'])
        staged.append((dest,raw))
    restored, skipped=0,0
    for dest,raw in staged:
        if dest.exists() and not overwrite:
            skipped+=1;continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        dest.write_bytes(raw);restored+=1
    print('복원:',restored,'건 / 기존 파일 보존:',skipped,'건')
    print('덮어쓰기를 원하면 먼저 별도 백업 후 overwrite=True로 명시하세요.')
    return restored,skipped

show_progress()


In [ ]:
#@title M02 환경 사전점검 (결과 기록, 비파괴) { display-mode: "form" }
from pathlib import Path
import ast
project_root = next((x for x in [Path('/content/QLT02_project'), Path('/QLT02_project'), Path.cwd()/'QLT02_project'] if x.is_dir()), Path('/content/QLT02_project'))
print('실습 폴더:', project_root, '존재:', project_root.is_dir())
if project_root.is_dir():
    print('미션별 양식:', (project_root/'forms'/'QLT02_M02_forms.md').exists())
    print('파이프라인:', (project_root/'pipeline'/'run_all.py').exists())
else:
    print('실습자료 자동 준비 셀을 먼저 실행하세요.')
print('이 검사는 학습 PASS나 교관 승인을 뜻하지 않습니다.')


## 📖 1. 왜 하는가? (상황·문제)


### 미션 개요

수령 검사 데이터 내의 측정치 결측 및 오류를 정제하고, 납품업체별 불량 이력을 정보 누수 없이 안전하게 인코딩하는 전처리 모듈을 구현합니다.

### 배경 및 필요성

원본 검사 데이터에는 중복 로트, 범위 오류, 측정치 결측 등 다양한 품질 저하 요인이 존재하므로, 후속 머신러닝 모델의 학습 정확도를 높이기 위해 엄격한 정제와 표준화가 필수적입니다. 특히 업체 불량 이력 계산 시 미래의 정보가 유출되지 않도록 통제하고, 군수 특수 예외 기록을 안전하게 보존하여 데이터의 신뢰성을 확보해야 합니다.

### 핵심 과업

전처리 규칙 적용 : 중복 로트, 업체 코드 표기, 날짜 형식, 재고번호 형식, 범위 오류 등 정제 규칙 10종을 적용합니다. 결측치 보간 : 치수 측정 결측치는 KNN(K-Nearest Neighbors) 알고리즘을 활용하여 정밀하게 보간합니다. 특수 예외 보존 및 이력 인코딩 : 긴급 불출 등 군수 특수 예외 기록은 삭제하지 않고 보존하며, 업체 불량 이력은 현재 로트 이전의 기록만으로 한정하여 정보 누수를 방지합니다.

### 검증 및 산출물

핵심 산출물 : 전처리 모듈 코드 ( src/etl/qlt02_data_cleansing.py ) 및 정제 품질 보고서 (전후 기술통계 비교 포함)

**평가 및 검증 기준 :**

정제 규칙 10종 적용 여부, 정합성 검증 통과율 100% 달성 여부, 특수 예외 기록 보존 여부, 그리고 업체 이력 인코딩 시 정보 누수 차단 여부

## 📖 2. 무엇을 해야 하는가? (학습 목표)


중복 로트, 업체 코드 표기, 날짜 형식, 재고번호 형식, 범위 오류 등 정제 규칙 10종을 적용하고 치수 측정 결측은 KNN으로 보간한다. 긴급 불출처럼 군수 특수 예외 기록은 삭제하지 않고 표시해 보존한다. 업체 불량 이력은 현재 로트 이전 기록만으로 계산하고, 정제 전후 기술통계와 정합성 검증 통과율을 비교한다.

### AI와 구현·검증할 항목

핵심 함수: clean. raw DataFrame → (정제 DataFrame, 처리 로그). 제공 정제 규칙과 학습 구간 기준의 결측치 처리를 구현한다. 추가 변경: KNN 이웃 수 3·5·10을 같은 학습/검증 조건에서 비교하고 공개 시점 기반 업체 이력의 누수 방지 검증을 실행합니다. 실제 AI 요청·답변·근거 검토3건·보완·재실행·최종 선택을 작성 양식에 기록합니다.

## 3. 준비 자료·제출물

- `src/etl/qlt02_data_cleansing.py`
- `reports/qlt02_cleansing_quality.md`
- `forms/QLT02_M02_forms.md`

## 📋 4. 학습활동별 수행 안내 및 평가 기준

아래 활동 1~4의 **수행 안내·AI 질문 예시·평가 조건**은 CMS에 등록한 해당 미션 내용과 같습니다.
코드 작성 및 실행은 아래의 별도 코드 셀에서 수행하고, 실제 AI 대화·분석 근거·산출물은 해당 미션의 `forms/QLT02_M02_forms.md`에 보관하세요.
Notebook에는 활동별 전용 입력 위젯이 없습니다. `save_mission_note()`와 Forms를 활용해 기록하며, 코드가 실행됐다는 이유만으로 학습 PASS 또는 교관 승인을 표시하지 않습니다.

### 활동 1 — 수행 범위와 검증 계획 정하기

### 활동 1. 수행 범위와 검증 계획 정하기

#### 이 활동에서 해결할 문제

원본 원장에서 중복 로트·형식 오류·치수 결측 등을 확인하고 정제 규칙 10종을 표로 만드세요.

#### 학생이 따라 할 순서

1. 해당 미션의 Lesson·Forms와 Student Notebook을 열어 제공된 가상 데이터와 원본 평가 기준을 확인합니다.
2. 원본 원장에서 중복 로트·형식 오류·치수 결측 등을 확인하고 정제 규칙 10종을 표로 만드세요. 긴급 불출 등 예외를 삭제하지 않도록 표시하고 업체 이력의 공개 시점을 사전에 정하세요.
3. 아래 질문은 예시입니다. 실제로 사용한 질문과 AI가 실제로 응답한 내용만 보존하고, AI의 제안은 원자료·실행 로그와 직접 대조합니다.
4. Student Notebook \*\*미션 2 활동 1\*\* 작성란에 본인 판단·원자료 근거·실제 AI 질문과 답변·수정 이유를 기록합니다.

#### AI 활용 예시

“가상 검사 원장에 적용할 데이터 정제 규칙 10가지를 표로 정리해 줘. 보존해야 할 예외와 현재 로트 이후에 공개된 업체 이력의 사용 금지를 별도로 점검해 줘.”

#### 확인할 결과와 기록

정제 규칙 10종·예외 목록·시점 기준과 원본 레코드 근거

자료에 없는 사실·성과·실행 시각을 만들지 않습니다. 실제 실행·정량 목표 충족·교관 승인은 각각 구분합니다.

#### 활동 1 — CMS 평가 기준

### 활동 1 — 수행 범위와 검증 계획 정하기 평가 기준

**PASS 판단 근거:** 10종 정제 규칙·보존할 특수 예외·업체 이력 공개 시점을 원본 원장과 연결했는가?

**확인 자료:** 정제 규칙 10종·예외 목록·시점 기준과 원본 레코드 근거. 실제 AI 요청·답변 원문과 학생이 직접 검토한 수정 전후 근거를 확인합니다.

**보완 필요:** 해당 활동의 정량 조건 미달, 원자료와 불일치, 코드/실행 기록 또는 대화 원문 누락, 시점 누수, 검증 없는 결과 확정.

**판정 원칙:** 미실시·미측정·미달은 PASS로 표시하지 않습니다. 저장·실행·학습 PASS·교관 승인과 실제 운영 승인은 서로 다릅니다.

### 활동 2 — 핵심 구현·첫 실행과 추가 변경

### 활동 2. 핵심 구현·첫 실행과 추가 변경

#### 이 활동에서 해결할 문제

clean 함수의 계약에 맞게 정제 코드와 로그를 구현하세요.

#### 학생이 따라 할 순서

1. 미션 학습자료의 Lesson·Forms 및 해당 Student Notebook을 열고, 앞 활동 결과를 확인합니다.
2. clean 함수의 계약에 맞게 정제 코드와 로그를 구현하세요. 첫 실행에서 각 규칙의 적용 건수·결측 처리·정합성 검사 출력을 보존하고, 잘못된 결과를 수정한 뒤 같은 데이터로 재실행하세요.
3. 아래 질문은 예시입니다. 실제로 사용한 질문과 AI가 실제로 응답한 내용만 보존하고, AI의 제안은 원자료·실행 로그와 직접 대조합니다.
4. Student Notebook \*\*미션 2 활동 2\*\* 작성란에 본인 판단·원자료 근거·실제 AI 질문과 답변·수정 이유를 기록합니다. 소스 코드 및 실행 결과는 지정된 프로젝트 경로에 저장하고 별도 백업합니다.

#### AI 활용 예시

“내 clean 함수와 실행 로그를 보여줄게. 규칙별 적용 건수, 반환 데이터 구조, 결측·오류 처리의 누락을 확인하고 코드의 최소 수정 방향을 알려줘.”

#### 확인할 결과와 기록

함수 구현·규칙별 로그·첫 실행과 수정 후 실행의 차이

자료에 없는 사실·성과·실행 시각을 만들지 않습니다. 실제 실행·정량 목표 충족·교관 승인은 각각 구분합니다.

#### 활동 2 — CMS 평가 기준

### 활동 2 — 핵심 구현·첫 실행과 추가 변경 평가 기준

**PASS 판단 근거:** clean이 정제 자료·로그를 반환하고10종 규칙 적용 건수·정합성 검사 통과율을 실제 출력으로 남겼는가?

**확인 자료:** 함수 구현·규칙별 로그·첫 실행과 수정 후 실행의 차이. 실제 AI 요청·답변 원문과 학생이 직접 검토한 수정 전후 근거를 확인합니다.

**보완 필요:** 해당 활동의 정량 조건 미달, 원자료와 불일치, 코드/실행 기록 또는 대화 원문 누락, 시점 누수, 검증 없는 결과 확정.

**판정 원칙:** 미실시·미측정·미달은 PASS로 표시하지 않습니다. 저장·실행·학습 PASS·교관 승인과 실제 운영 승인은 서로 다릅니다.

### 활동 3 — 근거 대조·보완·재실행

### 활동 3. 근거 대조·보완·재실행

#### 이 활동에서 해결할 문제

업체 이력에 현재 로트보다 먼저 공개된 과거 기록만 쓰는지 검사하세요.

#### 학생이 따라 할 순서

1. 미션 학습자료의 Lesson·Forms 및 해당 Student Notebook을 열고, 앞 활동 결과를 확인합니다.
2. 업체 이력에 현재 로트보다 먼저 공개된 과거 기록만 쓰는지 검사하세요. 교육용 30일 공개 가정과 KNN 보간 과정의 학습/검증 구간 분리를 점검하고 긴급 불출 예외 보존 여부를 비교하세요.
3. 아래 질문은 예시입니다. 실제로 사용한 질문과 AI가 실제로 응답한 내용만 보존하고, AI의 제안은 원자료·실행 로그와 직접 대조합니다.
4. Student Notebook \*\*미션 2 활동 3\*\* 작성란에 본인 판단·원자료 근거·실제 AI 질문과 답변·수정 이유를 기록합니다.

#### AI 활용 예시

“정제 전후 데이터와 업체 이력 계산 방식을 비교해 줘. 수령 시점에 공개되지 않은 결과의 누수, KNN 보간의 전체 데이터 참조, 긴급 불출 예외 삭제 위험을 찾아줘.”

#### 확인할 결과와 기록

업체 이력의 공개일·누수 검증·보존 예외 건수·전후 비교

자료에 없는 사실·성과·실행 시각을 만들지 않습니다. 실제 실행·정량 목표 충족·교관 승인은 각각 구분합니다.

#### 활동 3 — CMS 평가 기준

### 활동 3 — 근거 대조·보완·재실행 평가 기준

**PASS 판단 근거:** 긴급 불출 예외의 건수와 값이 보존되는지, 업체 이력이 현재 로트 이전에 공개된 기록만 쓰는지 검증했는가?

**확인 자료:** 업체 이력의 공개일·누수 검증·보존 예외 건수·전후 비교. 실제 AI 요청·답변 원문과 학생이 직접 검토한 수정 전후 근거를 확인합니다.

**보완 필요:** 해당 활동의 정량 조건 미달, 원자료와 불일치, 코드/실행 기록 또는 대화 원문 누락, 시점 누수, 검증 없는 결과 확정.

**판정 원칙:** 미실시·미측정·미달은 PASS로 표시하지 않습니다. 저장·실행·학습 PASS·교관 승인과 실제 운영 승인은 서로 다릅니다.

### 활동 4 — 최종 판단·산출물 제출

### 활동 4. 최종 판단·산출물 제출

#### 이 활동에서 해결할 문제

정제 전후 기술통계와 복원할 수 없는 결측·예외 조건을 보고서와 Forms에 적으세요.

#### 학생이 따라 할 순서

1. 미션 학습자료의 Lesson·Forms 및 해당 Student Notebook을 열고, 앞 활동 결과를 확인합니다.
2. 정제 전후 기술통계와 복원할 수 없는 결측·예외 조건을 보고서와 Forms에 적으세요. 10종 규칙과 실행 로그, AI 제안 중 반영/보류한 이유를 보존하고 결과물을 저장하세요.
3. 아래 질문은 예시입니다. 실제로 사용한 질문과 AI가 실제로 응답한 내용만 보존하고, AI의 제안은 원자료·실행 로그와 직접 대조합니다.
4. Student Notebook \*\*미션 2 활동 4\*\* 작성란에 본인 판단·원자료 근거·실제 AI 질문과 답변·수정 이유를 기록합니다.

#### AI 활용 예시

“정제 보고서와 로그를 붙일게. 10종 규칙의 실제 적용, 처리 전후 수치, 복원 한계, 원본 근거와 AI 수정 이력이 빠졌는지 확인해 줘.”

#### 확인할 결과와 기록

정제 보고서·코드·Forms·AI 대화·누락 및 미측정 표시

자료에 없는 사실·성과·실행 시각을 만들지 않습니다. 실제 실행·정량 목표 충족·교관 승인은 각각 구분합니다.

#### 활동 4 — CMS 평가 기준

### 활동 4 — 최종 판단·산출물 제출 평가 기준

**PASS 판단 근거:** 전후 기술통계·복원 한계·정제 보고서와 실제 AI 수정 기록을 제출했는가?

**확인 자료:** 정제 보고서·코드·Forms·AI 대화·누락 및 미측정 표시. 실제 AI 요청·답변 원문과 학생이 직접 검토한 수정 전후 근거를 확인합니다.

**보완 필요:** 해당 활동의 정량 조건 미달, 원자료와 불일치, 코드/실행 기록 또는 대화 원문 누락, 시점 누수, 검증 없는 결과 확정.

**판정 원칙:** 미실시·미측정·미달은 PASS로 표시하지 않습니다. 저장·실행·학습 PASS·교관 승인과 실제 운영 승인은 서로 다릅니다.


## 5. AI 요청 예시 (그대로 제출하지 말고 자신의 자료로 수정)

- lots_raw와 정제 규칙, clean 함수의 빈칸을 붙일게. 삭제·수정·보존할 기록을 구분하고 KNN 결측 보간을 학습 구간에서만 맞추는 방법을 설명해 줘. 긴급 불출 예외와 업체 이력의 공개 시점을 보존해 줘.

## 🔍 6. 검증·판정


### 평가 운영

원본 필수 미션. 전체사이클은10개모두수행합니다. 수행하는미션의정량·정성·진행/완료3항목은모두확인합니다. CMS의미션선택설정은별도확인합니다.

### PASS 확인 조건

10종 정제 규칙·보존할 특수 예외·업체 이력 공개 시점을 원본 원장과 연결했는가? clean이 정제 자료·로그를 반환하고10종 규칙 적용 건수·정합성 검사 통과율을 실제 출력으로 남겼는가? 긴급 불출 예외의 건수와 값이 보존되는지, 업체 이력이 현재 로트 이전에 공개된 기록만 쓰는지 검증했는가? 전후 기술통계·복원 한계·정제 보고서와 실제 AI 수정 기록을 제출했는가?

**원본기준:**

[정량적 평가][필수] 정제·정합성 검증  — 정제 규칙 10종 적용, 정합성 검증 통과율 100%, 특수 예외 기록 보존, 업체 이력 인코딩의 정보 누수 방지가 확인되었는가.

**PASS 증거**

clean이 정제 자료·로그를 반환하고10종 규칙 적용 건수·정합성 검사 통과율을 실제 출력으로 남겼는가? 긴급 불출 예외의 건수와 값이 보존되는지, 업체 이력이 현재 로트 이전에 공개된 기록만 쓰는지 검증했는가?

**보완 방법**

미달 지표는 실제 분모·기간·가정과 함께 표시하고 원인·수정 계획을 적은 뒤 재실행합니다. 목표를 달성했다고 수치를 바꾸지 않습니다.

**판정 메모**

달성·미달·추가확인필요와작성·실행·실제교관승인상태를구분합니다. 개인+AI를실제복수평가자/실무자검토로인정하지않습니다.

## 📁 7. 저장·제출 주의

실제 AI 요청과 답변 **원문**, 전후 코드, 실제 실행 결과, 근거 검토 3건 이상, 미달·미측정·미실시 구분, 본인 선택 이유를 미션 양식에 기록합니다. 코드가 실행됐다고 정량 목표를 충족하거나 교관이 승인한 것은 아닙니다.


## M02 검사 측정치 결측치 정제, 납품업체별 불량 이력 인코딩 전처리 모듈 구현

코드: `src/etl/qlt02_data_cleansing.py`

변경 과제: KNN 이웃 수 3·5·10을 같은 학습/검증 조건에서 비교하고 공개 시점 기반 업체 이력의 누수 방지 검증을 실행합니다.

기준 실행 후 파일을 수정하고 다시 실행합니다. 각 실행 결과는 새 폴더로 보존됩니다.

### 구현 계약

raw DataFrame → (정제 DataFrame, 처리 로그). 제공 정제 규칙과 학습 구간 기준의 결측치 처리를 구현한다.

빈 함수를 구현한 뒤 실행 셀로 넘어갑니다. 함수 정의의 모든 인자·기본값을 유지하세요.


In [ ]:
# 구현 후 주석을 해제해 저장하세요.
# implementation = """
# def clean(raw):
#     여기에 실제 구현을 작성
# """
# backup=replace_function(ROOT/'src/etl/qlt02_data_cleansing.py', 'clean', implementation)
# print("이전 코드 보존:", backup)


In [ ]:
from pipeline.run_all import run_mission
result=run_mission(2)
print("실행 상태(학습 PASS 아님):", result)

**M02 기록:** 실행 로그를 확인하고 `forms/QLT02_M02_forms.md`에 실제 AI 요청·답변, 근거 검토 3건 이상, 수정 전후 결과 및 본인 판단을 적으세요. 별도 메모는 `save_mission_note(2, "본인 판단", "실제 AI 대화 원문", "검증 근거")`로 저장할 수 있습니다. `show_progress()`는 실행 기록만 표시합니다.


## 이 미션 작업 상태와 임시 보관

먼저 실행 결과와 AI 대화를 원본 양식에 저장합니다. 아래 셀은 **검토용 ZIP만 생성**하며 합격·교관 승인을 뜻하지 않습니다.


In [ ]:
#@title M02 검토용 ZIP 만들기 (학생 실습 작업 보관)
from pathlib import Path
import zipfile, datetime
_r = Path(ROOT)
assert _r.is_dir(), "실습자료 자동 준비 셀부터 실행하세요."
_folder = _r.parent / "mission_review_archives"
_folder.mkdir(parents=True, exist_ok=True)
_out = _folder / "QLT02_M02_review.zip"
with zipfile.ZipFile(_out, "w", compression=zipfile.ZIP_DEFLATED) as _z:
    for _p in sorted(_r.rglob("*")):
        if _p.is_file() and "__pycache__" not in _p.parts:
            _z.write(_p, _p.relative_to(_r.parent))
print("검토용 ZIP:", _out, "(학습 PASS/교관 승인 아님)")
try:
    from google.colab import files
    files.download(str(_out))
except ImportError:
    print("Colab 외부 실행: 생성 경로에서 파일을 찾으세요.")
